### 파인튜닝이 끝난 모델 Ollama에 배포하기

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


### Ollama 에 배포된 모델 sageuk-qwen 사용

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "sageuk-qwen"


def ask_ollama(question: str) -> str:
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": question}
        ],
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0.7,
            "top_p": 0.8,
            "repeat_penalty": 1.12,
            "num_predict": 160,
        },
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=180,
    )

    # 오류가 있으면 여기서 에러 메시지를 보여줌
    response.raise_for_status()

    data = response.json()
    return data["message"]["content"].strip()


if __name__ == "__main__":
    question = "안녕하세요!"
    answer = ask_ollama(question)

    print("Q:", question)
    print("A:", answer)

In [ ]:
from ollama import Client, ResponseError

MODEL_NAME = "sageuk-qwen"
OLLAMA_HOST = "http://localhost:11434"

client = Client(host=OLLAMA_HOST)


def ask_ollama(question: str) -> str:
    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    try:
        response = client.chat(
            model=MODEL_NAME,
            messages=messages,
            stream=False,
            think=False,
            options={
                "temperature": 0.7,
                "top_p": 0.8,
                "repeat_penalty": 1.12,
                "num_predict": 160,
            },
        )

    except TypeError:
        # ollama-python 버전에 따라 think 인자를 아직 지원하지 않는 경우 대비
        response = client.chat(
            model=MODEL_NAME,
            messages=messages,
            stream=False,
            options={
                "temperature": 0.7,
                "top_p": 0.8,
                "repeat_penalty": 1.12,
                "num_predict": 160,
            },
        )

    # 최신 ollama-python은 response.message.content 접근 가능
    try:
        return response.message.content.strip()
    except AttributeError:
        # 구버전 / dict 방식 fallback
        return response["message"]["content"].strip()


if __name__ == "__main__":
    test_questions = [
        "안녕하세요!",
        "오늘 컨디션 어때?",
        "밥 먹었어?",
        "고마워요!",
        "요즘 스트레스가 심해",
        "주말에 뭐하지?",
    ]

    for q in test_questions:
        print("=" * 60)
        print("Q:", q)

        try:
            answer = ask_ollama(q)
            print("A:", answer)

        except ResponseError as e:
            print("Ollama 응답 오류:", e.error)

            if e.status_code == 404:
                print(f"모델 '{MODEL_NAME}'을 찾지 못했습니다.")
                print("먼저 아래 명령으로 등록 여부를 확인하세요:")
                print("ollama list")

        except Exception as e:
            print("기타 오류:", repr(e))

In [ ]:
from ollama import Client, ResponseError

MODEL_NAME = "sageuk-qwen"
OLLAMA_HOST = "http://localhost:11434"

client = Client(host=OLLAMA_HOST)


def extract_content(chunk) -> str:
    """
    ollama-python 버전에 따라 chunk가 dict처럼 오거나,
    객체처럼 오는 경우를 모두 처리합니다.
    """
    try:
        # dict 스타일
        return chunk["message"]["content"] or ""
    except Exception:
        pass

    try:
        # object 스타일
        return chunk.message.content or ""
    except Exception:
        return ""


def chat_once(history: list[dict], user_input: str) -> str:
    """
    사용자 입력 1개를 보내고, assistant 응답을 스트리밍으로 출력한 뒤
    최종 응답 문자열을 반환합니다.
    """
    messages = history + [
        {"role": "user", "content": user_input}
    ]

    options = {
        "temperature": 0.7,
        "top_p": 0.8,
        "repeat_penalty": 1.12,
        "num_predict": 160,
    }

    print("모델: ", end="", flush=True)

    answer_parts = []

    try:
        stream = client.chat(
            model=MODEL_NAME,
            messages=messages,
            stream=True,
            think=False,
            options=options,
        )
    except TypeError:
        # ollama-python 구버전에서 think 인자를 지원하지 않는 경우
        stream = client.chat(
            model=MODEL_NAME,
            messages=messages,
            stream=True,
            options=options,
        )

    for chunk in stream:
        text = extract_content(chunk)
        if text:
            print(text, end="", flush=True)
            answer_parts.append(text)

    print()

    return "".join(answer_parts).strip()


def main():
    history = []

    print("=" * 60)
    print(f"Ollama 대화 시작: {MODEL_NAME}")
    print("종료하려면 exit, quit, q 중 하나를 입력하세요.")
    print("=" * 60)

    while True:
        try:
            user_input = input("\n나: ").strip()

            if user_input.lower() in ["exit", "quit", "q"]:
                print("대화를 종료합니다.")
                break

            if not user_input:
                continue

            answer = chat_once(history, user_input)

            history.append({"role": "user", "content": user_input})
            history.append({"role": "assistant", "content": answer})

            # 너무 길어지는 것을 방지: 최근 20개 메시지만 유지
            if len(history) > 20:
                history = history[-20:]

        except KeyboardInterrupt:
            print("\n대화를 종료합니다.")
            break

        except ResponseError as e:
            print("\nOllama 응답 오류:", e)

            if getattr(e, "status_code", None) == 404:
                print(f"모델 '{MODEL_NAME}'을 찾지 못했습니다.")
                print("먼저 CMD에서 확인하세요:")
                print("ollama list")

        except Exception as e:
            print("\n오류 발생:", repr(e))


if __name__ == "__main__":
    main()